# ID10M BIO Diagnostic

Converts System E span predictions (char offsets) → token-level BIO tags.  
Computes token-level BIO macro F1 — directly comparable to ID10M paper Table 5.  
Also shows prediction distribution to catch classification collapse / label flip.

**Requires:** mBERT pred files downloaded from Drive:
- `id10m_system_e_english_preds.jsonl`
- `id10m_system_e_spanish_preds.jsonl`

And the original ID10M TSV files (already local).

In [6]:
import json
from pathlib import Path
from sklearn.metrics import f1_score, classification_report
import numpy as np

# ── CONFIG ────────────────────────────────────────────────────────────────────
PREDS_DIR = Path('/Users/shishirmaddineni/Desktop/Idiomator_Research/Research_And_Training/ID10m')  # EN+ES+HI+TE model preds from Drive
DATA_DIR  = Path('/Users/shishirmaddineni/Desktop/Idiomator_Research/Research_And_Training/ID10m')
LANGS     = ['EN', 'ES']
SPLIT     = 'test'

LANG_MAP = {
    'EN': ('English', 'english'),
    'ES': ('Spanish', 'spanish'),
}
PUNCT = set('.,;:!?)]\'\'"…—–')
print('Config OK')

Config OK


In [7]:
# ── LOAD GOLD TSV (token-level) ───────────────────────────────────────────────
def parse_bio_tsv_tokens(tsv_path):
    """Return list of (tokens, tags) per sentence — raw token-level gold."""
    sentences = []
    tokens, tags = [], []
    with open(tsv_path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if tokens:
                    sentences.append((tokens[:], tags[:]))
                tokens, tags = [], []
            else:
                parts = line.split('\t')
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    tags.append(parts[1].strip())
    if tokens:
        sentences.append((tokens, tags))
    return sentences

def reconstruct_sentence(tokens):
    s = ''
    for tok in tokens:
        if s and tok not in PUNCT and not tok.startswith("'"):
            s += ' '
        s += tok
    return s

gold_tsv = {}  # code -> list of (tokens, tags)
for code in LANGS:
    label, dirname = LANG_MAP[code]
    tsv = DATA_DIR / f'{SPLIT}_{dirname}.tsv'
    if not tsv.exists():
        print(f'[{code}] TSV not found: {tsv}')
        continue
    sents = parse_bio_tsv_tokens(tsv)
    gold_tsv[code] = sents
    print(f'[{code}] {len(sents)} sentences from TSV')

[EN] 200 sentences from TSV
[ES] 199 sentences from TSV


In [9]:
# ── LOAD mBERT PREDS ──────────────────────────────────────────────────────────
mbert_preds = {}
for code in LANGS:
    path = PREDS_DIR / f'id10m_system_e_{LANG_MAP[code][1]}_preds-2.jsonl'
    if not path.exists():
        print(f'[{code}] Pred file not found: {path}')
        continue
    preds = [json.loads(l) for l in path.open()]
    mbert_preds[code] = preds
    print(f'[{code}] {len(preds)} predictions loaded')

[EN] 200 predictions loaded
[ES] 199 predictions loaded


In [10]:
# ── DISTRIBUTION CHECK (catch label flip / collapse) ─────────────────────────
print('=== DISTRIBUTION CHECK ===')
for code, preds in mbert_preds.items():
    gold_dist = {k: sum(1 for p in preds if p['idiomaticity']     == k) for k in ('idiomatic','literal')}
    pred_dist = {k: sum(1 for p in preds if p['pred_idiomaticity']== k) for k in ('idiomatic','literal')}
    correct   = sum(1 for p in preds if p['idiomaticity'] == p['pred_idiomaticity'])
    print(f'\n[{code}]')
    print(f'  Gold : {gold_dist}')
    print(f'  Pred : {pred_dist}')
    print(f'  Accuracy: {correct}/{len(preds)} = {correct/len(preds):.3f}')
    print(f'  Sample predictions:')
    for p in preds[:8]:
        mark = '✓' if p['idiomaticity'] == p['pred_idiomaticity'] else '✗'
        print(f'    {mark} gold={p["idiomaticity"]:<10} pred={p["pred_idiomaticity"]:<10} | {p["sentence"][:60]}')

=== DISTRIBUTION CHECK ===

[EN]
  Gold : {'idiomatic': 159, 'literal': 41}
  Pred : {'idiomatic': 122, 'literal': 78}
  Accuracy: 149/200 = 0.745
  Sample predictions:
    ✓ gold=literal    pred=literal    | A  rock  has  broken  the  ice  covering  a  lake.
    ✓ gold=literal    pred=literal    | The  ship  broke  the  ice  all  the  way.
    ✓ gold=idiomatic  pred=idiomatic  | This  is  a  perfect  way  to  break  the  ice  and  start  
    ✓ gold=literal    pred=literal    | The  bullet  hit  thebook  in  his  pocket.
    ✓ gold=literal    pred=literal    | The  victim  was  stabbed  in  the  back  by  a  criminal.
    ✗ gold=literal    pred=idiomatic  | The  President  was  stabbed  in  the  back  while  talking 
    ✓ gold=literal    pred=literal    | We  rang  the  bell  but  nobody  answered.
    ✓ gold=literal    pred=literal    | I  thought  you  rang  the  bell.

[ES]
  Gold : {'idiomatic': 133, 'literal': 66}
  Pred : {'idiomatic': 85, 'literal': 114}
  Accuracy: 117/199 = 

In [11]:
# ── CONVERT CHAR SPANS → BIO TAGS ────────────────────────────────────────────
def char_span_to_bio(tokens, pred_char_s, pred_char_e):
    """Given token list + predicted char span, return BIO tag list."""
    # Reconstruct char offsets for each token
    tok_starts = []
    s = ''
    for tok in tokens:
        if s and tok not in PUNCT and not tok.startswith("'"):
            s += ' '
        tok_starts.append(len(s))
        s += tok
    tok_ends = [tok_starts[i] + len(tokens[i]) for i in range(len(tokens))]

    bio = []
    in_span = False
    for i, (ts, te) in enumerate(zip(tok_starts, tok_ends)):
        # Token overlaps with predicted span
        if pred_char_s is not None and pred_char_e is not None and ts < pred_char_e and te > pred_char_s:
            if not in_span:
                bio.append('B-IDIOM')
                in_span = True
            else:
                bio.append('I-IDIOM')
        else:
            bio.append('O')
            in_span = False
    return bio


TAG2ID = {'O': 0, 'B-IDIOM': 1, 'I-IDIOM': 2}

def compute_bio_f1(gold_tag_seqs, pred_tag_seqs):
    gold_flat = [TAG2ID[t] for seq in gold_tag_seqs for t in seq]
    pred_flat = [TAG2ID[t] for seq in pred_tag_seqs for t in seq]
    macro = f1_score(gold_flat, pred_flat, average='macro', zero_division=0)
    rep   = classification_report(
        gold_flat, pred_flat,
        labels=[0,1,2], target_names=['O','B-IDIOM','I-IDIOM'],
        output_dict=True, zero_division=0
    )
    return macro, rep

print('BIO conversion functions defined')

BIO conversion functions defined


In [12]:
# ── RUN BIO EVAL ──────────────────────────────────────────────────────────────
print('=== BIO TOKEN-LEVEL EVAL (comparable to ID10M paper Table 5) ===')

for code in LANGS:
    if code not in mbert_preds or code not in gold_tsv:
        print(f'[{code}] missing preds or TSV, skip')
        continue

    preds    = mbert_preds[code]
    tsv_sents = gold_tsv[code]

    if len(preds) != len(tsv_sents):
        print(f'[{code}] WARNING: pred count {len(preds)} != TSV count {len(tsv_sents)}')

    gold_bio_seqs = []
    pred_bio_seqs = []
    mismatches    = 0

    for i, (pred, (tokens, gold_tags)) in enumerate(zip(preds, tsv_sents)):
        # Verify sentence matches
        reconstructed = reconstruct_sentence(tokens)
        if reconstructed.strip() != pred['sentence'].strip():
            mismatches += 1
            if mismatches <= 3:
                print(f'  MISMATCH #{i}:')
                print(f'    TSV reconstruct : {reconstructed[:60]}')
                print(f'    Pred sentence   : {pred["sentence"][:60]}')

        # Gold BIO from TSV
        gold_bio_seqs.append(gold_tags)

        # Pred BIO from char span
        ps = pred.get('pred_span_start')
        pe = pred.get('pred_span_end')
        # If classified as literal, force all-O prediction
        if pred.get('pred_idiomaticity') == 'literal':
            pred_bio = ['O'] * len(tokens)
        else:
            pred_bio = char_span_to_bio(tokens, ps, pe)
        pred_bio_seqs.append(pred_bio)

    macro, rep = compute_bio_f1(gold_bio_seqs, pred_bio_seqs)

    print(f'\n[{code}]  sentence_mismatches={mismatches}')
    print(f'  BIO macro F1   : {macro:.4f}')
    print(f'  O         F1   : {rep["O"]["f1-score"]:.4f}  (support={int(rep["O"]["support"])})')
    print(f'  B-IDIOM   F1   : {rep["B-IDIOM"]["f1-score"]:.4f}  (support={int(rep["B-IDIOM"]["support"])})')
    print(f'  I-IDIOM   F1   : {rep["I-IDIOM"]["f1-score"]:.4f}  (support={int(rep["I-IDIOM"]["support"])})')

    # Span prediction sanity check
    n_zero_span = sum(1 for p in preds if p.get('pred_span_start') == 0 and p.get('pred_span_end') == 0)
    n_none_span = sum(1 for p in preds if p.get('pred_span_start') is None)
    print(f'  Span (0,0) preds: {n_zero_span}/{len(preds)}  (suspiciously high = span head broken)')
    print(f'  Span None  preds: {n_none_span}/{len(preds)}')

    # Sample span predictions
    print(f'  Sample span predictions (idiomatic only):')
    shown = 0
    for p in preds:
        if p['idiomaticity'] == 'idiomatic' and shown < 5:
            ps, pe = p.get('pred_span_start'), p.get('pred_span_end')
            gold_span = p['sentence'][p['span_start']:p['span_end']] if p.get('span_start') is not None else 'N/A'
            pred_span = p['sentence'][ps:pe] if ps is not None and pe is not None else 'NONE'
            print(f'    gold_span={gold_span!r:<20} pred_span={pred_span!r}')
            shown += 1

=== BIO TOKEN-LEVEL EVAL (comparable to ID10M paper Table 5) ===

[EN]  sentence_mismatches=0
  BIO macro F1   : 0.6147
  O         F1   : 0.9129  (support=2755)
  B-IDIOM   F1   : 0.4307  (support=159)
  I-IDIOM   F1   : 0.5006  (support=373)
  Span (0,0) preds: 0/200  (suspiciously high = span head broken)
  Span None  preds: 41/200
  Sample span predictions (idiomatic only):
    gold_span='break  the  ice '   pred_span='break  the  ice'
    gold_span='piece  of  cake'    pred_span='piece  of  cake'
    gold_span='under  the  weather ' pred_span='under  the  weather'
    gold_span='To  add  insult  to  the  injury' pred_span='went  away'
    gold_span='Break  a  leg '     pred_span='Break  a  leg'

[ES]  sentence_mismatches=0
  BIO macro F1   : 0.5896
  O         F1   : 0.9089  (support=1759)
  B-IDIOM   F1   : 0.3781  (support=133)
  I-IDIOM   F1   : 0.4818  (support=348)
  Span (0,0) preds: 0/199  (suspiciously high = span head broken)
  Span None  preds: 66/199
  Sample span predi